In [85]:
import os
import requests
import datetime as dt
import numpy as np
import pandas as pd
import polars as pl
import pandas_market_calendars as pcal
import yfinance as yf
# Set display options to show 30 rows
pl.Config.set_tbl_rows(30)  # For displaying DataFrames
# pl.Config.set_tbl_cols(-1)  # Show all columns

polars.config.Config

In [86]:
ticker = 'SPY.US'
token = os.environ.get('EODHD')
GET_LIVE_DATA = True
min_date = dt.datetime(2007, 1, 1)
max_date = dt.datetime(2026, 1, 1)

In [87]:
def get_bars_from_eodhd(ticker, min_date, max_date, token):
    import concurrent.futures

    def get_data(ticker, start_date, end_date, token, interval='1m', fmt='json'):
        start_ts = str(int(pd.to_datetime(start_date).timestamp()))
        end_ts = str(int(pd.to_datetime(end_date).timestamp()))

        url = f'https://eodhd.com/api/intraday/{ticker}?api_token={token}&fmt={fmt}&from={start_ts}&to={end_ts}&interval={interval}'
        response = requests.get(url)
        data = response.json()
        if len(data) == 0:
            print(f"No data for {start_date} to {end_date}")
            return
        df = pl.DataFrame(data)
        print(start_date, end_date, response.status_code, df.shape)
        df = df.with_columns(
            pl.col("datetime").str.to_datetime(format="%Y-%m-%d %H:%M:%S", time_zone="UTC").dt.convert_time_zone('US/Eastern')
        )

        if not os.path.exists(f"data/{ticker}"):
            os.makedirs(f"data/{ticker}")
        file_name = f"data/{ticker}/{ticker}_{start_date.strftime('%Y%m%d')}_{end_date.strftime('%Y%m%d')}.parquet"
        df.write_parquet(file_name)

        return
    dates = pl.date_ranges(start=min_date, end=max_date, interval="120d", eager=True)[0]
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for index, date in enumerate(dates[:-1]):
            future = executor.submit(get_data, ticker, dates[index], dates[index + 1], token)
            futures.append(future)

        for future in concurrent.futures.as_completed(futures):
            pass

def get_time_diff(df):
    time_change = df.with_columns(
        pl.col("time").diff().alias("time_diff"),
        pl.col("date").shift(1).alias("prev_date")
    ).with_columns(
        pl.when(pl.col("date") == pl.col("prev_date"))
        .then(pl.col("time_diff"))
        .otherwise(pl.lit(None))
        .alias("same_day_time_diff")
    )

    time_diff_counts = time_change.filter(
        pl.col("same_day_time_diff").is_not_null()
    ).group_by("same_day_time_diff").agg(
        pl.len().alias("count")
    ).sort("count", descending=True)

    # Display the results
    time_diff_counts = time_diff_counts.with_columns(
        (pl.col("count") / time_change.shape[0] * 100).round(2).alias("percentage")
    )
    return time_change, time_diff_counts

def process_raw_files(ticker):
    ### Concatenate the downloaded files into one

    df2 = pl.scan_parquet(f'data/{ticker}/*{ticker}*.parquet')
    df2 = df2.select("datetime", "open", "high", "low", "close", "volume")
    # split datetime into time and date
    df2 = df2.with_columns(pl.col('datetime').dt.time().alias('time'))
    df2 = df2.with_columns(pl.col('datetime').dt.date().alias('date'))
    df2 = df2.sort("datetime", descending=False).collect()

    ### Exclude after-hours trading. For this need to pull the calendar for NYSE
    # NOTE: Obviously need to change if the ticker being processed does not trade NYSE hours
    nyse = pcal.get_calendar("NYSE")
    h = nyse.schedule(start_date=min_date, end_date=max_date)
    ix = pcal.date_range(h, frequency='1m', closed='both')
    ix = pl.Series(ix).dt.convert_time_zone('US/Eastern').alias('actual_datetime')
    ix = ix.dt.cast_time_unit('us').to_frame()
    ix = ix.with_columns(pl.lit(True).alias('is_open'))

    df2 = df2.join(
        ix,
        left_on='datetime',
        right_on='actual_datetime',
        how='left')
    df2 = df2.with_columns(pl.col('is_open').fill_null(False))
    df2 = df2.filter(pl.col('is_open'))

    time_change, _ = get_time_diff(df2)

    holidays = time_change.filter(pl.col('same_day_time_diff') == pd.Timedelta(hours=2, minutes=20)).select('date').to_series().to_list()
    df2 = df2.filter(~pl.col('date').is_in(holidays) | (pl.col('date').is_in(holidays) & (pl.col('time').dt.time() <= dt.time(13, 0))))

    return df2


def get_and_save_daily_from_eodhd(ticker, min_date, max_date, token):
    import yfinance as yf
    url = f'https://eodhd.com/api/eod/{ticker}?from={min_date.strftime("%Y-%m-%d")}&to={max_date.strftime("%Y-%m-%d")}&period=d&api_token={token}&fmt=json'
    response = requests.get(url)
    data = response.json()
    df_daily = pl.DataFrame(data)
    df_daily = df_daily.with_columns(pl.col('date').str.strptime(pl.Date, '%Y-%m-%d'))
    
    vix = yf.download('^VIX', start=min_date.date(), end=max_date.date())
    vix = pl.DataFrame(vix.loc[:, ('Close', '^VIX')].rename('VIX').to_frame().reset_index())
    vix = vix.with_columns(pl.col('Date').cast(pl.Date)).rename({'Date': 'date'})
    df_daily = df_daily.rename({'adjusted_close': ticker}).select(['date', ticker]).join(pl.DataFrame(vix), how='left', on='date')
    full_file = f"data/{ticker}.daily.parquet"
    if os.path.exists(full_file):
        os.remove(full_file)
    df_daily.write_parquet(full_file)
    return df_daily

def get_and_save_divs_from_eodhd(ticker, min_date, max_date, token):
    url = f'https://eodhd.com/api/div/{ticker}?from={min_date.strftime("%Y-%m-%d")}&to={max_date.strftime("%Y-%m-%d")}&period=d&api_token={token}&fmt=json'
    response = requests.get(url)
    data = response.json()
    df_div = pl.DataFrame(data)
    df_div = df_div.select('paymentDate', 'value').rename({'value': ticker, 'paymentDate': 'date'})
    df_div = df_div.with_columns(pl.col('date').cast(pl.Date))

    full_file = f"data/{ticker}.div.parquet"
    if os.path.exists(full_file):
        os.remove(full_file)
    df_div.write_parquet(full_file)
    return df_div    

In [88]:
if GET_LIVE_DATA:
    get_bars_from_eodhd(ticker=ticker, min_date=min_date, max_date=max_date, token=token)
    get_and_save_daily_from_eodhd(ticker=ticker, min_date=min_date, max_date=max_date, token=token)
    get_and_save_divs_from_eodhd(ticker=ticker, min_date=min_date, max_date=max_date, token=token)
df2 = process_raw_files(ticker)

No data for 2009-04-20 to 2009-08-18
No data for 2011-12-06 to 2012-04-04
No data for 2009-08-18 to 2009-12-16
No data for 2007-12-27 to 2008-04-25
No data for 2007-01-01 to 2007-05-01
No data for 2007-05-01 to 2007-08-29
No data for 2008-04-25 to 2008-08-23
No data for 2012-08-02 to 2012-11-30
No data for 2007-08-29 to 2007-12-27
No data for 2008-12-21 to 2009-04-20
No data for 2012-04-04 to 2012-08-02
No data for 2010-04-15 to 2010-08-13
No data for 2011-08-08 to 2011-12-06
No data for 2009-12-16 to 2010-04-15
No data for 2011-04-10 to 2011-08-08
No data for 2010-08-13 to 2010-12-11
No data for 2008-08-23 to 2008-12-21
No data for 2010-12-11 to 2011-04-10
No data for 2012-11-30 to 2013-03-30
No data for 2013-03-30 to 2013-07-28
2013-07-28 2013-11-25 200 (39549, 8)
2013-11-25 2014-03-25 200 (42136, 8)
2014-03-25 2014-07-23 200 (45920, 8)
2014-11-20 2015-03-20 200 (45988, 8)
2015-11-15 2016-03-14 200 (55654, 8)
2017-11-04 2018-03-04 200 (58615, 8)
2015-07-18 2015-11-15 200 (44033, 8)
2

[*********************100%***********************]  1 of 1 completed


### Quick Sanity Checks

In [89]:
# null checks
total = df2.select('datetime', 'open', 'high', 'low', 'close').null_count().sum().row(0)
assert (sum(v for v in total if v is not None) == 0)

# dupe observation checks
assert sum(df2.select(pl.col('datetime').is_duplicated()).sum().row(0)) == 0

### Value Delta Checks
The remaining volatility seems legit. It's on days with known events like Flash Crash in 2010, fears over Chinese economy in Aug 2015.

In [90]:
# Calculate percentage change for all numeric columns
numeric_cols = ['open', 'close', 'high', 'low']

# Calculate pct_change for all numeric columns
pct_changes = df2.select(
    [pl.col(col).pct_change().alias(f"{col}_pct_change")
     for col in numeric_cols]
)

# Get top 5 changes (positive or negative) for each column
for col in numeric_cols:
    col_name = f"{col}_pct_change"
    print(f"\nTop 5 changes for {col}:")
    top_changes = pct_changes.select(
        pl.col(col_name).abs().sort(descending=True).head(5)
    ).to_series().to_list()

    # Get the original rows with these changes
    top_rows = df2.filter(
        pl.col(col).pct_change().abs().is_in(top_changes)
    ).select(
        [pl.col(col).pct_change().alias("pct_change"), pl.all()]
    ).sort(
        pl.col("pct_change").abs(), 
        descending=True
    ).head(5)

    print(top_rows.to_pandas())


Top 5 changes for open:
   pct_change                  datetime    open    high     low    close  \
0         NaN 2020-03-09 09:30:00-04:00  275.29  276.24  275.05  276.030   
1   -0.070072 2020-03-12 09:30:00-04:00  256.00  257.50  255.12  256.320   
2   -0.057891 2020-03-16 09:30:00-04:00  241.18  241.19  240.76  240.890   
3   -0.020441 2020-03-18 09:30:00-04:00  236.25  239.01  236.22  238.205   

    volume      time       date  is_open  
0  2444783  09:30:00 2020-03-09     True  
1  3270422  09:30:00 2020-03-12     True  
2   961129  09:30:00 2020-03-16     True  
3  2500808  09:30:00 2020-03-18     True  

Top 5 changes for close:
   pct_change                  datetime    open    high     low   close  \
0         NaN 2015-08-24 09:30:00-04:00  187.33  187.64  185.69  186.00   
1    0.484032 2020-03-09 09:30:00-04:00  275.29  276.24  275.05  276.03   
2   -0.071405 2020-03-12 09:30:00-04:00  256.00  257.50  255.12  256.32   
3   -0.060198 2020-03-16 09:30:00-04:00  241.18  241.

### Write Concatenated Results

In [91]:
full_file = f"data/{ticker}.parquet"

if os.path.exists(full_file):
    os.remove(full_file)
df2 = df2.select('datetime', 'date', 'time', 'open', 'high', 'low', 'close', 'volume')
df2.write_parquet(f"data/{ticker}.parquet")